In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)


In [ ]:
reds = pd.read_csv("winequality-red.csv", sep=";")
whites = pd.read_csv("winequality-white.csv", sep=";")

wines = pd.concat([reds, whites], ignore_index=True)

wines

In [ ]:
features = wines.columns.drop("quality").to_list()
label = "quality"
features

In [ ]:
wines.info()

In [ ]:
wines_discrete_quality = wines.copy(deep=True)
wines[label] = wines[label].apply(lambda quality: 1 if quality >= 6 else -1)

# wine counts separated by the original quality
counts = wines_discrete_quality[label].value_counts().sort_index()
norm = wines_discrete_quality[label].value_counts(normalize=True).sort_index()
print(f"Number of wines for each quality value:")
for quality in counts.index:
    count = f"quality {quality} -> {counts[quality]} wines"
    percent = f"({round(norm[quality]*100, 2)}%)"
    print(f"{count:<24}{percent:>15}")

# wine counts separated in the two classes GOOD (1) and BAD (-1)
counts = wines[label].value_counts().sort_index()
norm = wines[label].value_counts(normalize=True).sort_index()
print(f"\nNumber of good (1) and bad (-1) wines:")
for quality in counts.index:
    count = f"quality {quality} -> {counts[quality]} wines"
    percent = f"({round(norm[quality]*100, 2)}%)"
    print(f"{count:<24}{percent:>14}")

In [ ]:
desc = wines.describe().T
desc["skew"] = wines.skew(axis=0)
desc['kurtosis'] = wines.kurtosis(axis=0)
desc.drop(columns='count', inplace=True)
desc

In [ ]:
display(wines.value_counts().loc[lambda x : x > 1])
display(wines.value_counts(subset=features).loc[lambda x : x > 1])

In [ ]:
for feature in features:
    fig = plt.figure(figsize=(17, 5))
    fig.suptitle(feature)
    hist = plt.subplot(1, 2, 1)
    hist.hist(x=wines[feature], bins='scott', orientation='horizontal')
    hist.set_ylabel(feature)
    hist.set_xlabel("Frequencies")
    
    box = plt.subplot(1, 2, 2, sharey=hist)
    box.boxplot(x=wines[feature])
    box.set_ylabel(feature)
    box.set_xticks([])

    plt.show(fig)
    plt.close(fig)


In [ ]:
def outlier_limits(df: pd.DataFrame, kind):
    lower_limit = df.min()
    upper_limit = df.max()
    if kind == "iqr":
        q1 = df.quantile(0.25)
        q3 = df.quantile(0.75)
        iqr = q3 - q1
        lower_limit = q1 - 1.5*iqr
        upper_limit = q3 + 1.5*iqr
    if kind == "zscore":
        mean = df.mean()
        std = df.std()
        lower_limit = mean - 3*std
        upper_limit = mean + 3*std
    if kind == "percentile":
        lower_limit = df.quantile(0.01)
        upper_limit = df.quantile(0.99)
    return lower_limit, upper_limit

for type in ['iqr', 'zscore', 'percentile']:
    lb, ub = outlier_limits(wines[features], type)
    outlier_condition = (wines[features] < lb) | (wines[features] > ub)
    out_per_feature = outlier_condition.sum()
    total_num_of_rows_with_outliers = wines[features][(outlier_condition)].any(axis=1).sum()
    print(f"outliers for {type}:")
    print(out_per_feature)
    print(f"total number of rows with an outlier: {total_num_of_rows_with_outliers} -> {round((total_num_of_rows_with_outliers / len(wines))*100, 2)}% of the total\n")


In [ ]:
lb, ub = outlier_limits(wines[features], 'iqr')
clipped = wines[features].clip(lb, ub, axis=1)
clipped[label] = wines[label]

for feature in features:
    fig = plt.figure(figsize=(17, 5))
    fig.suptitle(feature)
    hist = plt.subplot(1, 2, 1)
    hist.hist(x=clipped[feature], bins='scott', orientation='horizontal')
    hist.set_ylabel(feature)
    hist.set_xlabel("Frequencies")
    
    box = plt.subplot(1, 2, 2, sharey=hist)
    box.boxplot(x=clipped[feature])
    box.set_ylabel(feature)
    box.set_xticks([])

    plt.show(fig)
    plt.close(fig)


In [ ]:
#for f_x, f_y in enumerate(features)
l = len(features)
for i in range(l):
    for j in range(i+1, l):
        sns.scatterplot(  
            x=features[i], 
            y=features[j], 
            data=clipped,
            hue=label
        )
        plt.show()
        plt.close()

In [ ]:
correlation = clipped[features].corr()
display(correlation)
sns.heatmap(correlation, annot=True)